In [ ]:
import os, math, time
import numpy as np
import pandas as pd
from scipy.io import loadmat, savemat
from pathlib import Path
from typing import List, Union
from tqdm import tqdm
from numba import njit, prange

# ========= 可配置参数 =========
CSV_PATH        = "ps_test.csv"
XSTABLE_W_PATH  = "test_all_xstable_and_w.mat"   # 提供 all_xstable, all_w（每组参数的稳态与权重）

S = 0.5
n_hill = 4

n_cells       = 50_000
time_interval = 0.1
T             = 100.0
STEADY_FRACTION = 0.8  # R4：只保留后20%窗口(0.8T,T]，不保存0.8T本身
delta         = 0.01
sigma         = 0.1  # 初值的高斯扰动（非系统噪声强度）

# 输出到当前运行目录下的子文件夹（Jupyter通常为本Notebook所在目录）
OUTPUT_ROOT = Path.cwd().resolve()
output_folder_traj   = OUTPUT_ROOT / f"trajectory_results_ps_test_n_cells={n_cells}"
output_folder_density= OUTPUT_ROOT / f"density_results_ps_test_n_cells={n_cells}"
SAVE_TRAJECTORY      = False  # True 则同时保存轨迹
SAVE_DTYPE = np.float32

# ========= 工具函数（兼容 MATLAB cell）=========
def _mat_cell_to_list(arr) -> List:
    if isinstance(arr, np.ndarray) and arr.dtype == object:
        flat = []
        if arr.ndim == 2:
            n0, n1 = arr.shape
            if n0 == 1 or n1 == 1:
                for i in range(n0*n1):
                    r = i // n1
                    c = i % n1
                    flat.append(arr[r, c])
            else:
                for i in range(arr.size):
                    flat.append(arr.flat[i])
        elif arr.ndim == 1:
            flat = list(arr)
        else:
            flat = list(arr.flat)
        return flat
    return [arr]

def _ensure_2d(a: np.ndarray) -> np.ndarray:
    return a[None, :] if a.ndim == 1 else a

def _coerce_weights_uniform(w: Union[np.ndarray, list]) -> np.ndarray:
    w = np.array(w, dtype=float).ravel()
    if w.size <= 1:
        return np.array([1.0], dtype=float)
    return np.ones_like(w) / w.size

# ========= 加载数据 =========
t0_load = time.perf_counter()

ps_test = pd.read_csv(CSV_PATH)
Times = len(ps_test)

mxw = loadmat(XSTABLE_W_PATH, squeeze_me=True)
if "all_xstable" not in mxw or "all_w" not in mxw:
    raise KeyError("mat 文件缺 'all_xstable' 或 'all_w'")

all_xstable_list = _mat_cell_to_list(mxw["all_xstable"])
all_w_list       = _mat_cell_to_list(mxw["all_w"])

if not (len(all_xstable_list) == len(all_w_list) == Times):
    raise ValueError(
        f"数量不匹配：Times={Times}, len(xstable)={len(all_xstable_list)}, len(w)={len(all_w_list)}"
    )

Path(output_folder_traj).mkdir(parents=True, exist_ok=True)
Path(output_folder_density).mkdir(parents=True, exist_ok=True)

# 用整数积分步确定采样时刻，避免浮点累加造成边界偏移。
assert T > 0 and delta > 0 and time_interval > 0, "时间参数必须为正"
assert 0 <= STEADY_FRACTION < 1, "稳态起始比例必须位于[0,1)"
n_steps = int(round(T / delta))
steady_step = int(round(STEADY_FRACTION * T / delta))
store_stride = int(round(time_interval / delta))
assert math.isclose(n_steps * delta, T, rel_tol=0.0, abs_tol=1e-10), "T必须是delta的整数倍"
assert math.isclose(steady_step * delta, STEADY_FRACTION * T, rel_tol=0.0, abs_tol=1e-10), "稳态窗口起点必须对齐积分步"
assert store_stride > 0 and math.isclose(store_stride * delta, time_interval, rel_tol=0.0, abs_tol=1e-10), "time_interval必须是delta的正整数倍"
assert n_steps > steady_step and (n_steps - steady_step) % store_stride == 0, "稳态窗口必须包含整数个采样间隔"
n_points = (n_steps - steady_step) // store_stride
sample_times = np.arange(steady_step + store_stride, n_steps + 1, store_stride, dtype=np.int64) * delta
assert len(sample_times) == n_points

print(f"加载完成，用时 {time.perf_counter()-t0_load:.2f}s | Times={Times}, n_cells={n_cells}, "
      f"采样窗口=({STEADY_FRACTION*T:g},{T:g}]，每条轨迹{n_points}点，"
      f"首末时刻={sample_times[0]:g}/{sample_times[-1]:g}，每组共{n_cells*n_points:,}个状态点")

# ---- 双基因 Hill 函数（Numba 友好）----
@njit(cache=True, fastmath=True)
def Ha(x, S, n):  # 自激活
    return (x**n) / (S**n + x**n)

@njit(cache=True, fastmath=True)
def Hr(x, S, n):  # 互抑制
    return (S**n) / (x**n + S**n)

# ---- 目标系统的确定性漂移项 f(x) ----
@njit(cache=True, fastmath=True)
def two_gene_drift(x, a, b, k, S, n, F_out):
    # x = [X1, X2]
    F_out[0] = a*Ha(x[0], S, n) + b*Hr(x[1], S, n) - k*x[0]
    F_out[1] = a*Ha(x[1], S, n) + b*Hr(x[0], S, n) - k*x[1]

# ---- 采样权重的二分查找 ----
@njit(cache=True)
def _search_cdf(cdf, u):
    lo = 0
    hi = cdf.size - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if u <= cdf[mid]:
            hi = mid
        else:
            lo = mid + 1
    return lo

# ---- 从稳态集合采样正初值 ----
@njit(cache=True, fastmath=True)
def _pick_init_positive(stable_states, cdf, sigma, out):
    d = stable_states.shape[1]
    while True:
        idx = _search_cdf(cdf, np.random.rand())
        for j in range(d):
            out[j] = np.random.normal(stable_states[idx, j], sigma)
        ok = True
        for j in range(d):
            if out[j] <= 0.0:
                ok = False; break
        if ok:
            return

# ---- 主仿真：双基因 + 噪声强度 D ----
@njit(parallel=True, cache=True, fastmath=True)
def simulate_cells_numba(
    n_cells, n_steps, steady_step, store_stride, delta,
    a, b, k, S, n_hill, D,
    stable_states, cdf, sigma,
    out_X, out_t
):
    sqrt_delta = math.sqrt(delta)
    for cell in prange(n_cells):
        x = np.empty(2, dtype=np.float64)
        _pick_init_positive(stable_states, cdf, sigma, x)
        F = np.empty(2, dtype=np.float64)

        # 从0积分到T，只存储后20%窗口中的状态；不保存初值或窗口左端点。
        store_i = 0
        for step in range(1, n_steps + 1):
            two_gene_drift(x, a, b, k, S, n_hill, F)

            # R4的Euler–Maruyama更新：dXi = fi dt + sqrt(2D) dWi。
            # 两个独立标准正态增量；非负边界仍按下方反射处理。
            xi1, xi2 = np.random.randn(2)
            gw1 = sqrt_delta * math.sqrt(2.0 * D) * xi1
            gw2 = sqrt_delta * math.sqrt(2.0 * D) * xi2

            x0 = x[0] + delta*F[0] + gw1
            x1 = x[1] + delta*F[1] + gw2

            # MATLAB 里使用 abs 保证为正
            if x0 < 0.0: x0 = -x0
            if x1 < 0.0: x1 = -x1
            x[0], x[1] = x0, x1

            if step > steady_step and (step - steady_step) % store_stride == 0:
                out_X[cell, store_i, 0] = np.float32(x[0])
                out_X[cell, store_i, 1] = np.float32(x[1])
                out_t[cell, store_i]    = np.float32(step * delta)
                store_i += 1

# ---- 读取参数并批量仿真 + 计算密度 ----
# ---- 读取参数并批量仿真 + 计算密度 ----
outer_iter = range(1, Times+1)

for NN in tqdm(outer_iter, desc="参数组(CPU)"):
    # 取一行参数（用 iloc 防止 FutureWarning）
    row = ps_test.iloc[NN-1]
    a, b, k, D = map(float, row.iloc[[0, 1, 2, 3]])

    # 稳态：必须为 2 维（X1, X2）
    stable = np.array(all_xstable_list[NN-1], dtype=float)
    stable = _ensure_2d(stable)
    if stable.shape[1] < 2:
        raise ValueError("稳态维度 < 2；此脚本为双基因模型，需要二维稳态。")
    if stable.shape[1] > 2:
        stable = stable[:, :2]
    stable = np.ascontiguousarray(stable.astype(np.float64))

    # 权重 -> CDF
    w = _coerce_weights_uniform(all_w_list[NN-1]).astype(np.float64)
    cdf = np.cumsum(w); cdf[-1] = 1.0
    cdf = np.ascontiguousarray(cdf)

    # 结果容器（轨迹）
    X_all_rd_cells = np.zeros((n_cells, n_points, 2), dtype=SAVE_DTYPE)
    time_all_cells = np.zeros((n_cells, n_points),   dtype=SAVE_DTYPE)

    # 仿真
    t0 = time.perf_counter()
    simulate_cells_numba(
        n_cells, n_steps, steady_step, store_stride, delta,
        a, b, k, S, n_hill, D,
        stable, cdf, sigma,
        X_all_rd_cells, time_all_cells
    )
    dt = time.perf_counter()-t0
    print(f"NN={NN} CPU 模拟完成，用时 {dt:.2f}s")

    # （可选）保存轨迹
    if SAVE_TRAJECTORY:
        traj_file = os.path.join(output_folder_traj, f"ps_test{{{NN}}}.mat")
        savemat(traj_file, {
            "X_all_rd_cells": X_all_rd_cells,
            "time_all_cells": time_all_cells,
            # 如需轨迹文件也存参数，取消下一行注释
            # "parameter": np.array([[a, b, k, D]], dtype=SAVE_DTYPE),
        }, do_compression=True)
        print(f"已保存轨迹到 {traj_file}")

    # ========= 仅用后20%窗口的状态计算 density, Xedges, Yedges =========
    X1_all_cells = X_all_rd_cells[:, :, 0].T.reshape(-1)
    X2_all_cells = X_all_rd_cells[:, :, 1].T.reshape(-1)
    x = X1_all_cells.astype(np.float64)
    y = X2_all_cells.astype(np.float64)

    # 边界与步长（对齐 0.05 网格）
    lim_up = max(float(x.max()), float(y.max()))
    lim_bottom = min(float(x.min()), float(y.min()), 0.0)
    lim_bottom = math.floor(lim_bottom * 20.0) / 20.0
    lim_up     = math.ceil (lim_up     * 20.0) / 20.0

    step = 0.05
    edges = np.arange(lim_bottom, lim_up + step + 1e-12, step, dtype=np.float64)

    H, Xedges, Yedges = np.histogram2d(x, y, bins=[edges, edges])

    total = H.sum()
    density = H / total if total > 0 else H

    # 1x4 参数矩阵
    parameter = np.array([[a, b, k, D]], dtype=SAVE_DTYPE)

    # 保存 density 结果（包含参数）
    dens_file = os.path.join(output_folder_density, f"ps_test{{{NN}}}.mat")
    savemat(
        dens_file,
        {
            "density": density.astype(SAVE_DTYPE),
            "sample_times": sample_times,  # 80.1至100.0，共200个时刻
            "sampling_window": np.array([[STEADY_FRACTION*T, T]], dtype=np.float64),  # 左开右闭
            "sample_count_per_cell": np.array([[n_points]], dtype=np.int64),
            "Xedges":  Xedges.astype(SAVE_DTYPE),
            "Yedges":  Yedges.astype(SAVE_DTYPE),
            "parameter": parameter,   # <--- 新增
        },
        do_compression=True
    )
    print(f"已保存密度到 {dens_file}")
